# 🖊️ 한글 손글씨 폰트 생성기 (DM-Font)
**28글자 손글씨 → 전체 한글 폰트 자동 생성**

NAVER CLOVA DM-Font (ECCV 2020) — 한글 손글씨 전용 사전학습 모델 사용

---
### ⚠️ 시작 전
상단 메뉴 → **런타임 → 런타임 유형 변경 → T4 GPU** 선택

In [ ]:
# ✅ Cell 1 — GPU 확인
import torch
print('GPU:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU 모델:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('GPU가 없습니다. 런타임 유형을 T4 GPU로 변경하세요.')

In [ ]:
# ✅ Cell 2 — DM-Font 클론 & 의존성 설치
import os

if not os.path.exists('dmfont'):
    !git clone https://github.com/clovaai/dmfont.git
%cd dmfont

!pip install -q h5py sconf
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install -q Pillow numpy tqdm
!apt-get install -qq potrace fontforge python3-fontforge

print('✅ 설치 완료')

In [ ]:
# ✅ Cell 3 — 한글 손글씨 사전학습 가중치 다운로드
import os
os.makedirs('checkpoints', exist_ok=True)

if not os.path.exists('checkpoints/korean-handwriting.pth'):
    !wget -q --show-progress \
        https://github.com/clovaai/dmfont/releases/download/v1.0.0/korean-handwriting.pth \
        -O checkpoints/korean-handwriting.pth

size = os.path.getsize('checkpoints/korean-handwriting.pth') / 1e6
print(f'✅ 가중치 다운로드 완료: {size:.1f} MB')

In [ ]:
# ✅ Cell 4 — 28글자 목록 확인
# 자모 커버리지가 최대인 28글자
REF_CHARS = [
    '갈', '넓', '읽', '좋', '밝', '흙', '닭', '삶',
    '젊', '긁', '뚫', '얽', '훑', '핥', '굶', '잃',
    '닳', '끓', '떫', '볶', '뒤', '쥐', '뢰', '쾌',
    '나', '다', '라', '마'
]

print(f'수집할 글자 ({len(REF_CHARS)}자):')
print(' '.join(REF_CHARS))
print()
print('위 글자들을 A4 종이에 손으로 쓰고 스캔하세요.')
print('파일명: 갈.png, 넓.png, ... (한글 음절 그대로)')

In [ ]:
# ✅ Cell 5 — 손글씨 이미지 업로드 (28장)
from google.colab import files
import os

os.makedirs('ref_images', exist_ok=True)
print('갈.png, 넓.png ... 28개 파일을 선택하세요.')

uploaded = files.upload()
for fname, data in uploaded.items():
    with open(f'ref_images/{fname}', 'wb') as f:
        f.write(data)

found = [f for f in os.listdir('ref_images') if f.endswith('.png')]
print(f'\n✅ {len(found)}장 업로드 완료')
if len(found) < 28:
    print(f'⚠️ {28 - len(found)}장 부족 — 더 업로드하려면 이 셀을 다시 실행하세요')

In [ ]:
# ✅ Cell 6 — 이미지 전처리 & 미리보기
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import os

os.makedirs('ref_preprocessed', exist_ok=True)

def preprocess(path, size=128):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    _, binary = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    coords = cv2.findNonZero(binary)
    if coords is not None:
        x, y, w, h = cv2.boundingRect(coords)
        pad = 12
        x, y = max(0,x-pad), max(0,y-pad)
        w = min(binary.shape[1]-x, w+2*pad)
        h = min(binary.shape[0]-y, h+2*pad)
        binary = binary[y:y+h, x:x+w]
    sz = max(binary.shape)
    out = np.zeros((sz,sz), dtype=np.uint8)
    ph, pw = (sz-binary.shape[0])//2, (sz-binary.shape[1])//2
    out[ph:ph+binary.shape[0], pw:pw+binary.shape[1]] = binary
    return cv2.resize(out, (size,size), interpolation=cv2.INTER_AREA)

src_files = sorted([f for f in os.listdir('ref_images') if f.endswith('.png')])
results = []
for fname in src_files:
    char = fname.replace('.png','')
    processed = preprocess(f'ref_images/{fname}')
    cv2.imwrite(f'ref_preprocessed/{fname}', processed)
    results.append((char, processed))

# 미리보기
cols = min(7, len(results))
rows = (len(results) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols*1.5, rows*1.5))
axes = np.array(axes).flat
for ax, (char, img) in zip(axes, results):
    ax.imshow(img, cmap='gray')
    ax.set_title(char, fontsize=10)
    ax.axis('off')
for ax in list(axes)[len(results):]:
    ax.axis('off')
plt.tight_layout()
plt.show()
print(f'✅ {len(results)}장 전처리 완료')

In [ ]:
# ✅ Cell 7 — HDF5 데이터셋 빌드 (DM-Font 입력 포맷)
import h5py
import numpy as np
import cv2
import json
import os

FONT_NAME = 'my_handwriting'
IMG_SIZE = 128

src_files = sorted([f for f in os.listdir('ref_preprocessed') if f.endswith('.png')])
chars = [f.replace('.png','') for f in src_files]

# HDF5 생성
with h5py.File('ref_data.hdf5', 'w') as h5:
    grp = h5.create_group(FONT_NAME)
    for fname in src_files:
        char = fname.replace('.png','')
        img = cv2.imread(f'ref_preprocessed/{fname}', cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        # DM-Font 포맷: 흰 배경 → 255, 글자 → 0 (반전)
        img = 255 - img
        grp.create_dataset(hex(ord(char))[2:].upper(), data=img)

# 메타 JSON
char_hexes = [hex(ord(c))[2:].upper() for c in chars]
meta = {FONT_NAME: char_hexes}
with open('ref_meta.json', 'w') as f:
    json.dump(meta, f)

print(f'✅ HDF5 생성: {len(chars)}자 저장됨')
print(f'   글자: {" ".join(chars)}')

In [ ]:
# ✅ Cell 8 — DM-Font 추론 실행
import sys
sys.path.insert(0, '.')

os.makedirs('output/png', exist_ok=True)

# 생성할 전체 한글
ALL_KOREAN = [chr(c) for c in range(0xAC00, 0xD7A4)]
gen_chars = [hex(ord(c))[2:].upper() for c in ALL_KOREAN]

# gen_chars.json 저장
with open('gen_chars.json', 'w') as f:
    import json
    json.dump(gen_chars, f)

# DM-Font evaluator 실행
!python evaluator.py \
    my_handwriting \
    checkpoints/korean-handwriting.pth \
    output/png \
    cfgs/kor.yaml \
    --mode user-study-save \
    --data_path ref_data.hdf5 \
    --ref_chars ref_meta.json

generated = len([f for f in os.listdir('output/png') if f.endswith('.png')])
print(f'\n✅ 생성 완료: {generated}자')

In [ ]:
# ✅ Cell 9 — 결과 미리보기 (랜덤 36자)
import os, random
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

png_dir = 'output/png'
all_pngs = [f for f in os.listdir(png_dir) if f.endswith('.png')]
samples = random.sample(all_pngs, min(36, len(all_pngs)))

fig, axes = plt.subplots(6, 6, figsize=(12, 12))
for ax, fname in zip(axes.flat, samples):
    img = Image.open(os.path.join(png_dir, fname)).convert('L')
    ax.imshow(img, cmap='gray')
    char = fname.replace('.png', '')
    ax.set_title(char, fontsize=14)
    ax.axis('off')
plt.suptitle('생성된 손글씨 샘플', fontsize=16)
plt.tight_layout()
plt.savefig('output/preview.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ output/preview.png 저장됨')

In [ ]:
# ✅ Cell 10 — PNG → SVG → TTF 변환
import subprocess
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
import os

os.makedirs('output/svg', exist_ok=True)

png_files = [f for f in os.listdir('output/png') if f.endswith('.png')]
print(f'벡터화 중... ({len(png_files)}자)')

errors = []
for fname in tqdm(png_files):
    char = fname.replace('.png', '')
    png_path = f'output/png/{fname}'
    bmp_path = f'/tmp/{char}.bmp'
    svg_path = f'output/svg/{char}.svg'
    if os.path.exists(svg_path):
        continue
    img = Image.open(png_path).convert('L')
    arr = np.array(img)
    binary = (arr < 128).astype(np.uint8) * 255
    Image.fromarray(binary).convert('1').save(bmp_path)
    r = subprocess.run(['potrace', bmp_path, '-s', '-o', svg_path, '--tight'],
                       capture_output=True)
    if r.returncode != 0:
        errors.append(char)

svg_count = len([f for f in os.listdir('output/svg') if f.endswith('.svg')])
print(f'\nSVG 완료: {svg_count}자')

# SVG → TTF
print('TTF 빌드 중...')
try:
    import fontforge
    font = fontforge.font()
    font.fontname = 'MyHandwriting'
    font.fullname = 'My Handwriting'
    font.familyname = 'My Handwriting'
    font.encoding = 'UnicodeFull'
    font.em = 1000

    for svg_file in tqdm(os.listdir('output/svg')):
        if not svg_file.endswith('.svg'):
            continue
        char = svg_file.replace('.svg', '')
        if len(char) != 1:
            continue
        glyph = font.createChar(ord(char))
        glyph.width = 1000
        try:
            glyph.importOutlines(f'output/svg/{svg_file}')
            glyph.correctDirection()
        except:
            pass

    font.generate('output/my_handwriting.ttf')
    size = os.path.getsize('output/my_handwriting.ttf') / 1e6
    print(f'\n✅ TTF 완료: output/my_handwriting.ttf ({size:.1f} MB)')
except ImportError:
    print('fontforge 모듈 없음 — 아래 셀의 대체 방법 사용')

In [ ]:
# ✅ Cell 11 — 다운로드
from google.colab import files
import os

for path, label in [
    ('output/my_handwriting.ttf', 'TTF 폰트'),
    ('output/preview.png', '미리보기')
]:
    if os.path.exists(path):
        files.download(path)
        print(f'✅ {label} 다운로드 시작: {path}')
    else:
        print(f'⚠️ {label} 없음: {path}')